# Vehicle Counting and Traffic Flow Analysis Using OpenCV

## Overview

This project extends a computer vision-based vehicle detection system by implementing vehicle tracking and directional counting using Python and OpenCV.

The system processes traffic footage to identify moving vehicles within a defined Region of Interest (ROI). A virtual counting line is used to record vehicles travelling towards downtown, while temporary tracking identifiers help prevent duplicate counts.

The implementation analyses two traffic videos and calculates the total number of counted vehicles and their average traffic flow rate in cars per minute.

## Technologies Used

- Python
- OpenCV
- NumPy
- Matplotlib
- Jupyter Notebook

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import math

## Loading and Inspecting Traffic Footage

The first step involves loading the traffic video using OpenCV and displaying an initial frame for visual inspection.

This allows the traffic scene to be examined before processing begins and helps identify the road area where vehicles will be detected and counted.

The initial frame also provides a reference for defining the Region of Interest (ROI) used in subsequent processing.

In [ ]:
cap = cv2.VideoCapture("Traffic_Laramie_1.mp4")

ret, frame = cap.read()
print("Frame from original video :")

plt.figure(figsize = (10,6))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

cap.release()

## Region of Interest (ROI) Selection

A Region of Interest (ROI) is defined to focus vehicle detection and counting on the relevant road area.

The ROI boundaries are calculated relative to the original frame dimensions, allowing a selected portion of the traffic scene to be extracted for processing.

This reduces unnecessary background information and helps concentrate the detection process on vehicles travelling along the road.

The extracted ROI is displayed to verify that the appropriate area has been selected.

In [ ]:
#Setting ROI 
h, w, _ = frame.shape

shift = int(w * 0.10) 
y1 = int(h * 0.35)
y2 = int(h * 0.95)
x1 = int(w * 0.05) + shift
x2 = int(w * 0.95) + shift
x1 = max(0, x1)
x2 = min(w, x2)

roi = frame[y1:y2, x1:x2]

plt.figure(figsize=(10,6))
plt.imshow(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
plt.axis("off")
print("ROI preview : ")
plt.show()

## Vehicle Tracking and Directional Counting

The `count_cars_to_downtown()` function implements a vehicle counting system using motion detection, centroid-based tracking, and a virtual counting line.

### Detection and Tracking Process

The function processes each video frame through the following stages:

1. **Frame Preprocessing:** Extracts the Region of Interest (ROI) and applies Gaussian blur to reduce image noise.
2. **Background Subtraction:** Uses the MOG2 algorithm to identify moving foreground objects.
3. **Morphological Filtering:** Applies opening and closing operations to reduce noise and improve the foreground mask.
4. **Contour Detection:** Identifies moving objects and filters out contours below a minimum area threshold.
5. **Centroid Tracking:** Calculates the centre coordinates of each detected object and associates detections with existing tracks based on Euclidean distance.
6. **Directional Counting:** Uses a horizontal counting line to register vehicles moving upwards across the selected region.
7. **Duplicate Count Prevention:** Marks tracks as counted after crossing the line to avoid counting the same tracked object repeatedly.

### Traffic Flow Calculation

The system calculates two traffic measurements:

- **Total Vehicle Count:** Number of tracked objects recorded crossing the virtual counting line.
- **Cars per Minute:** Total counted vehicles divided by the processed video duration in minutes.

The function returns both values for further comparison and analysis.

In [ ]:
#function for displaying and counting
def count_cars_to_downtown(video_path, y1, y2, x1, x2,
                          line_y=120, min_area=800,
                          dist_thresh=50, max_miss=10,
                          show=True):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)

    bg = cv2.createBackgroundSubtractorMOG2(
        history=500, varThreshold=50, detectShadows=False
    )
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5,5))

    tracks = {}
    next_id = 0
    count_up = 0
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1

        roi = frame[y1:y2, x1:x2]
        roi_blur = cv2.GaussianBlur(roi, (5,5), 0)

        fg = bg.apply(roi_blur)
        _, fg = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)
        fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, kernel, iterations=1)
        fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, kernel, iterations=2)

        contours, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        detections = []
        for cnt in contours:
            if cv2.contourArea(cnt) < min_area:
                continue
            x, y, w, h = cv2.boundingRect(cnt)
            cx = x + w//2
            cy = y + h//2
            detections.append((cx, cy))

        # mark all tracks as missed
        for tid in list(tracks.keys()):
            tracks[tid]["miss"] += 1

        # match detections to tracks
        for (cx, cy) in detections:
            best_id = None
            best_dist = 1e9

            for tid, t in tracks.items():
                dist = math.hypot(cx - t["cx"], cy - t["cy"])
                if dist < best_dist:
                    best_dist = dist
                    best_id = tid

            if best_dist < dist_thresh:
                # update existing track
                t = tracks[best_id]
                t["prev_cy"] = t["cy"]
                t["cx"] = cx
                t["cy"] = cy
                t["miss"] = 0
            else:
                # new track
                tracks[next_id] = {
                    "cx": cx, "cy": cy,
                    "prev_cy": cy,
                    "counted": False,
                    "miss": 0
                }
                next_id += 1

        # remove stale tracks
        for tid in [tid for tid, t in tracks.items() if t["miss"] > max_miss]:
            del tracks[tid]

        if show:
            view = roi.copy()
            cv2.line(view, (0, line_y), (view.shape[1], line_y), (255, 0, 0), 2)

        # count crossings
        for tid, t in tracks.items():
            cy = t["cy"]
            prev_cy = t["prev_cy"]

            if (not t["counted"]) and (prev_cy > line_y) and (cy <= line_y):
                count_up += 1
                t["counted"] = True

            # update prev for next loop
            t["prev_cy"] = cy

            if show:
                cv2.circle(view, (t["cx"], t["cy"]), 3, (0,255,0), -1)

        if show:
            cv2.putText(view, f"Cars to Downtown: {count_up}", (10,30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
            cv2.imshow(f"Counting: {video_path}", view)
            if cv2.waitKey(30) & 0xFF == 27:
                break

    cap.release()
    if show:
        cv2.destroyAllWindows()

    minutes = (frame_count / fps) / 60 if fps > 0 else 0
    cars_per_min = count_up / minutes if minutes > 0 else 0

    return count_up, cars_per_min


## Traffic Video Analysis

The vehicle counting function is applied to two traffic videos to compare their traffic flow characteristics.

Both videos are processed using the same detection and tracking approach, with a virtual counting line positioned at `LINE_Y = 120` within the Region of Interest.

For each video, the system calculates:

- **Total Vehicle Count:** Number of tracked objects crossing the counting line in the specified direction.
- **Cars per Minute:** Average number of counted vehicles per minute of processed footage.

These measurements provide a basis for comparing the observed traffic flow between the two video sequences.

In [ ]:
LINE_Y = 120

total1, cpm1 = count_cars_to_downtown(
    "Traffic_Laramie_1.mp4", y1, y2, x1, x2,
    line_y=LINE_Y, show=True
)

total2, cpm2 = count_cars_to_downtown(
    "Traffic_Laramie_2.mp4", y1, y2, x1, x2,
    line_y=LINE_Y, show=True
)

print("Traffic_Laramie_1.mp4:", total1, "cars,", cpm1, "cars/min")
print("Traffic_Laramie_2.mp4:", total2, "cars,", cpm2, "cars/min")


## Vehicle Counting Strategy

Vehicles are counted by analysing their movement within the selected Region of Interest (ROI).

A virtual horizontal counting line is placed across the road. When a tracked object crosses this line in the upward direction, the system registers a count.

Each detected object is assigned a temporary tracking identifier, and a counted flag is used to prevent repeated counts for the same track.

The resulting vehicle count is used to calculate the average traffic flow rate in cars per minute, based on the number of processed frames and the video's frame rate.

## Traffic Flow Comparison

The results from both traffic videos are presented in a comparison table.

The table displays the total number of counted vehicles and the calculated traffic flow rate in cars per minute.

For presentation purposes, the cars-per-minute values are rounded down to whole numbers using Python's `math.floor()` function.

This comparison provides an overview of the observed vehicle movement in each video.

In [ ]:
# Data from your results
cpm1_rd = math.floor(cpm1)
cpm2_rd = math.floor(cpm2)

data = [
    ["Video", "Cars", "Cars/Min (Round down)"],
    ["Traffic_Laramie_1", total1, cpm1_rd],
    ["Traffic_Laramie_2", total2, cpm2_rd]
]

plt.figure()
plt.axis("off")

table = plt.table(cellText=data, loc="center")
table.scale(1, 2)

plt.title("Traffic Flow Comparison")
plt.show()


## Results and Limitations

### Results

The implementation demonstrates a vehicle counting system using background subtraction, centroid-based tracking, and directional line crossing detection.

The system processes two traffic video sequences and calculates the total number of counted vehicles and their average traffic flow rate in cars per minute.

The results are presented in a comparison table to illustrate differences in observed traffic flow between the two videos.

### Limitations

- **Tracking Accuracy:** The nearest-centroid tracking approach may incorrectly associate vehicles when multiple objects overlap or move close together.
- **Detection Sensitivity:** Background subtraction can be affected by shadows, lighting changes, and environmental movement.
- **Counting Accuracy:** Missed detections or incorrect tracking associations may lead to inaccurate vehicle counts.
- **Camera Orientation:** The counting direction is based on upward movement within the video frame and assumes that this corresponds to traffic travelling towards downtown.
- **Parameter Sensitivity:** Detection thresholds, tracking distances, and the counting line position may require adjustment for different traffic footage.

### Input Requirements

The notebook requires two traffic video files:

- `Traffic_Laramie_1.mp4`
- `Traffic_Laramie_2.mp4`

These videos are not included in the GitHub repository. Compatible traffic footage can be supplied, although the ROI coordinates and detection parameters may need adjustment.

### Conclusion

This project demonstrates how basic computer vision techniques can be combined to estimate directional traffic flow from video footage.

It extends motion-based vehicle detection by incorporating temporary object tracking, line-crossing detection, and traffic flow calculations.